# MNIST MLP3: AdamW vs Local-Delta ECS WW-PGD

This notebook runs five baseline seeds and five local-delta ECS extension seeds on MLP3-MNIST. The extension wraps `adamw` and, at the end of every epoch, fractionally damps the completed epoch displacement outside the current ECS instead of reshaping the full weight matrix spectrum.

## Install/check optional diagnostics

WeightWatcher is optional at runtime. If it is missing, the experiment still records fallback SVD diagnostics: rank-slope alpha proxy, local ECS rank, and trace-log residual.

In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys

for import_name, pip_name in {
    "weightwatcher": "weightwatcher>=0.7.7",
}.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Optional dependency {pip_name} is not installed. The notebook will use fallback SVD diagnostics unless you install it.")

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
package_root = None
for root in [cwd, *cwd.parents]:
    direct = root / "wwpgd_local_delta"
    nested = root / "optimizers" / "wwpgd_local_delta" / "wwpgd_local_delta"
    if direct.is_dir():
        package_root = root
        break
    if nested.is_dir():
        package_root = nested.parent
        break
if package_root is None:
    raise RuntimeError("Could not find wwpgd_local_delta. Run this notebook from the optimizer folder or repository root.")
if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))
print("Using package root:", package_root)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)


def plot_metric(df, metric, title, ylabel):
    fig, ax = plt.subplots(figsize=(9, 5))
    for arm, frame in df.groupby("arm"):
        grouped = frame.groupby("epoch")[metric].agg(["mean", "std"]).reset_index()
        x = grouped["epoch"].to_numpy()
        mean = grouped["mean"].to_numpy()
        std = grouped["std"].fillna(0.0).to_numpy()
        ax.plot(x, mean, marker="o", label=arm)
        ax.fill_between(x, mean - std, mean + std, alpha=0.15)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


def plot_layer_metric(df, metric, title, ylabel):
    if metric not in df.columns:
        print(f"Skipping {metric}: column not present")
        return
    clean = df.copy()
    clean[metric] = pd.to_numeric(clean[metric], errors="coerce")
    clean = clean.dropna(subset=[metric])
    if clean.empty:
        print(f"Skipping {metric}: no numeric values")
        return
    for layer, layer_df in clean.groupby("layer_name"):
        fig, ax = plt.subplots(figsize=(9, 5))
        for arm, frame in layer_df.groupby("arm"):
            grouped = frame.groupby("epoch")[metric].agg(["mean", "std"]).reset_index()
            x = grouped["epoch"].to_numpy()
            mean = grouped["mean"].to_numpy()
            std = grouped["std"].fillna(0.0).to_numpy()
            ax.plot(x, mean, marker="o", label=arm)
            ax.fill_between(x, mean - std, mean + std, alpha=0.15)
        ax.set_title(f"{title}: {layer}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()


def plot_corrections(corrections):
    if corrections.empty:
        print("No correction rows recorded.")
        return
    for metric, ylabel in [
        ("orthogonal_fraction", "||Delta_perp|| / ||Delta||"),
        ("removed_fraction_of_base", "||removed|| / ||Delta||"),
        ("ecs_rank", "ECS rank"),
        ("trace_log_per_eval", "Trace-log residual"),
    ]:
        if metric not in corrections.columns:
            continue
        plot_layer_metric(corrections.rename(columns={"parameter": "layer_name"}), metric, f"Local-delta correction {metric}", ylabel)

## Configure five paired runs

The arms are `baseline` and `local_delta_ecs`. Each uses the same seed list. The local-delta correction is applied once per epoch with `correction_fraction=0.25`.

In [ ]:
from wwpgd_local_delta import MNISTRunConfig
from wwpgd_local_delta.mnist_experiment import run_mnist_comparison, summarize_final_performance

CONFIG = MNISTRunConfig(
    optimizer_kind="adamw",
    epochs=10,
    seeds=(1337, 2027, 4099, 7919, 104729),
    correction_fraction=0.25,
    apply_every_epochs=1,
    warmup_epochs=0,
    normalization_gamma=0.0,
    data_dir="./data",
    output_dir="./runs_adamw_local_delta_ecs",
    ww_enabled=True,
)
CONFIG

In [ ]:
result = run_mnist_comparison(CONFIG, progress=True)
result.save(CONFIG.output_dir)
print("Saved outputs to", Path(CONFIG.output_dir).resolve())

## Tables

In [ ]:
display(summarize_final_performance(result.performance))
display(result.performance.tail(20))
display(result.spectral.tail(20))
display(result.corrections.tail(20))

## Plots

In [ ]:
plot_metric(result.performance, "train_acc", "Training accuracy", "Accuracy")
plot_metric(result.performance, "test_acc", "Test accuracy", "Accuracy")
plot_metric(result.performance, "train_loss", "Training cross-entropy", "Loss")
plot_metric(result.performance, "test_loss", "Test cross-entropy", "Loss")

# WeightWatcher alpha when available. The fallback SVD alpha proxy is also plotted.
plot_layer_metric(result.spectral, "alpha", "WeightWatcher alpha / fallback alpha", "alpha")
plot_layer_metric(result.spectral, "alpha_proxy", "Rank-slope alpha proxy", "alpha proxy")
plot_layer_metric(result.spectral, "ERG_gap", "WeightWatcher ERG gap", "ERG gap")
plot_layer_metric(result.spectral, "ecs_rank_local", "Local self-consistent ECS rank", "rank")
plot_layer_metric(result.spectral, "trace_log_per_eval_local", "Local ECS trace-log residual", "trace log")

plot_corrections(result.corrections)